In [1]:
import torchvision
from torchvision import datasets
from pathlib import Path
from torch.utils.data import DataLoader
import os
import torch
from torch import nn 
from torch.utils.tensorboard import SummaryWriter
from tqdm.auto import tqdm

In [2]:
vit_weights = torchvision.models.ViT_B_32_Weights.DEFAULT
vit_transforms = vit_weights.transforms()

In [3]:
vit_transforms

ImageClassification(
    crop_size=[224]
    resize_size=[256]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BILINEAR
)

In [4]:
vit_data_dir = Path("vit_data_augmented")

vit_train_transforms = torchvision.transforms.Compose([
    torchvision.transforms.TrivialAugmentWide(),
    torchvision.transforms.RandomRotation(degrees=30),
    torchvision.transforms.RandomHorizontalFlip(p=0.5),
    vit_transforms 
])

# Getting ViT traning data
vit_train_data = datasets.FGVCAircraft(root=vit_data_dir,
                                       split="train",
                                       transform=vit_train_transforms,
                                       download=True)

# Get ViT test data
vit_test_data = datasets.FGVCAircraft(root=vit_data_dir,
                                      split="test",
                                      transform=vit_transforms,
                                      download=True)

100%|██████████| 2753340328/2753340328 [04:17<00:00, 10672624.99it/s]


Extracting vit_data_augmented/fgvc-aircraft-2013b.tar.gz to vit_data_augmented


In [5]:
NUM_WORKERS = os.cpu_count()
BATCH_SIZE = 32 

vit_train_dataloder = DataLoader(dataset=vit_train_data,
                                 batch_size=BATCH_SIZE,
                                 shuffle=True,
                                 num_workers=NUM_WORKERS,
                                 pin_memory=True)

vit_test_dataloder = DataLoader(dataset=vit_test_data,
                                 batch_size=BATCH_SIZE,
                                 shuffle=True,
                                 num_workers=NUM_WORKERS,
                                 pin_memory=True)

In [6]:
device = "mps" if torch.backends.mps.is_available() else "cpu"
device

'mps'

In [39]:
def create_vit(device,
               num_classes: int = 100,
               seed: int = 42):
    weights = torchvision.models.ViT_B_16_Weights.DEFAULT
    model = torchvision.models.vit_b_16(weights=weights).to(device)

    # Freeze feature extraction layers
    for param in model.parameters():
        param.requires_grad = False

    torch.manual_seed(seed)
    torch.mps.manual_seed(seed)
    # Create a new classifier layer
    
    model.heads = nn.Sequential(
        nn.Linear(in_features=768, out_features=num_classes),
    )

    return model

In [8]:
num_classes = len(vit_train_data.classes)
num_classes

100

In [9]:
def train(model: torch.nn.Module,
          writer,
          train_dataloader: torch.utils.data.DataLoader,
          test_dataloader:  torch.utils.data.DataLoader,
          loss_fn: torch.nn.Module,
          optimizer: torch.optim.Optimizer,
          device,
          model_input_size: torch.randn, #torch.randn(32, 1, 28, 28)
          epochs: int = 10,
          save_model: bool = False,
          save_model_path: str = "./models",
          model_name: str = "vit_model"):
    
    for epoch in tqdm(range(epochs)):
        # Put the model in train mode
        model.to(device)
        model.train()

        # Setup train loss and train accuracy values
        train_loss, train_acc = 0, 0

        for batch, (X, y) in enumerate(train_dataloader):
            # Send data to target device
            X, y = X.to(device), y.to(device)

            # Forward pass
            y_pred = model(X)

            # Calculate and accumulate loss 
            loss = loss_fn(y_pred, y)
            train_loss += loss.item()

            # Optimizer zero grad
            optimizer.zero_grad()

            # Loss backward
            loss.backward()

            # Optimizer step
            optimizer.step()

            # Calculate and accumulate accuray metrics across all batches
            y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)
            train_acc += (y_pred_class == y).sum().item()/len(y_pred)

        # Adjust metrics to get average loss and accuracy per batch
        train_loss = train_loss / len(train_dataloader)
        train_acc = train_acc / len(train_dataloader)

        # Put the model in eval mode
        model.to(device)
        model.eval()

        # Setup test loss and test accuracy values
        test_loss, test_acc = 0, 0

        with torch.inference_mode():
            # Loop through DataLoader batches
            for batch, (X, y) in enumerate(test_dataloader):
                # Send data to target deivce
                X, y = X.to(device), y.to(device)

                # Forward pass
                test_pred_logits = model(X)

                # Calculate and accumulate loss
                loss = loss_fn(test_pred_logits, y)
                test_loss += loss.item()

                # Calculate and accumulate accuracy
                test_pred_labels = test_pred_logits.argmax(dim=1) 
                test_acc += ((test_pred_labels == y).sum().item() / len(test_pred_labels))

            # Adjust metrics to get average loss and accuracy per batch
            test_loss = test_loss / len(train_dataloader)
            test_acc = test_acc / len(test_dataloader)
            
            results = {"train_loss": [],
                        "train_acc": [],
                        "test_loss": [],
                        "test_acc": []}
            print(
                    f"Epoch: {epoch+1} | "
                    f"train_loss: {train_loss:.4f} | "
                    f"train_acc: {train_acc:.4f} | "
                    f"test_loss: {test_loss:.4f} | "
                    f"test_acc: {test_acc:.4f}")
            results["train_loss"].append(train_loss)
            results["train_acc"].append(train_acc)
            results["test_loss"].append(test_loss)
            results["test_acc"].append(test_acc)
            writer.add_scalar(tag="Train Loss",
                              scalar_value=train_loss,
                              global_step=epoch)

            writer.add_scalar(tag="Train Accuracy",
                              scalar_value=train_acc,
                              global_step=epoch)             

            writer.add_scalar(tag="Test Loss",
                              scalar_value=test_loss,
                              global_step=epoch) 
            writer.add_scalar(tag="Test Accuracy",
                              scalar_value=test_acc,
                              global_step=epoch) 
            writer.add_graph(model=model,
                             input_to_model=model_input_size.to(device))
    if save_model == True:
        print(f"[INFO] Saving {model_name} model to {save_model_path}")
        MODEL_PATH = Path(save_model_path) 
        MODEL_PATH.mkdir(parents=True,
                         exist_ok=True)
        MODEL_SAVE_PATH = MODEL_PATH/model_name
        torch.save(obj=model.state_dict(),
                   f=MODEL_SAVE_PATH)
        torch.mps.empty_cache()
        return results
    else:
        torch.mps.empty_cache()
        return results

In [38]:
vit_100_epochs = create_vit(device=device,
                           num_classes=num_classes,
                           seed=42)

vit_100_epochs.to(device)

hundred_epochs_loss_fn = torch.nn.CrossEntropyLoss()

hundred_epochs_optimizer = torch.optim.Adam(params=vit_100_epochs.parameters(),
                             lr=0.001)

writer = SummaryWriter(log_dir=os.path.join("./new_results", "100_epochs"))

train(model=vit_100_epochs,
      train_dataloader=vit_train_dataloder,
      writer=writer,
      model_input_size=torch.randn(32,3,224,224),
      test_dataloader=vit_test_dataloder,
      loss_fn=hundred_epochs_loss_fn,
      optimizer=hundred_epochs_optimizer,
      device=device,
      epochs=100,
      save_model=True,
      save_model_path="./new_models",
      model_name="100_epochs")

TypeError: empty(): argument 'size' must be tuple of ints, but found element of type float at pos 2

In [47]:
test = create_vit(device=device, num_classes=num_classes, seed=42).to(device)
test.eval()
print(torch.argmax(torch.softmax(test(vit_train_data[1][0].unsqueeze(dim=0).to(device)), dim=1), dim=1))

tensor([73], device='mps:0')


In [37]:
vit_train_data[0][0].unsqueeze(dim=0).shape

torch.Size([1, 3, 224, 224])

In [2]:
"""
import torch
import torchvision
from PIL import Image

# Load the model
model = torchvision.models.resnet50(pretrained=True)

# Load the test data
test_data = torchvision.datasets.ImageFolder('path/to/test/data')

# Create a list to store the incorrect images
incorrect_images = []

# Iterate over the test data
for image, target in test_data:

    # Convert the image to a PyTorch tensor
    image = torch.from_numpy(image).float()

    # Make a prediction
    output = model(image)

    # Get the predicted class
    predicted_class = torch.argmax(output)

    # If the prediction is incorrect, add the image to the list
    if predicted_class != target:
        incorrect_images.append(image)

# Save the list of incorrect images
torch.save(incorrect_images, 'incorrect_images.pt')
"""

"\nimport torch\nimport torchvision\nfrom PIL import Image\n\n# Load the model\nmodel = torchvision.models.resnet50(pretrained=True)\n\n# Load the test data\ntest_data = torchvision.datasets.ImageFolder('path/to/test/data')\n\n# Create a list to store the incorrect images\nincorrect_images = []\n\n# Iterate over the test data\nfor image, target in test_data:\n\n    # Convert the image to a PyTorch tensor\n    image = torch.from_numpy(image).float()\n\n    # Make a prediction\n    output = model(image)\n\n    # Get the predicted class\n    predicted_class = torch.argmax(output)\n\n    # If the prediction is incorrect, add the image to the list\n    if predicted_class != target:\n        incorrect_images.append(image)\n\n# Save the list of incorrect images\ntorch.save(incorrect_images, 'incorrect_images.pt')\n"